# Adversarial Testing

اختبار رسائل الهجوم المولّدة ضد موديل الدفاع المدرّب.

> **ملاحظة:** هذا النوتبوك يعمل **بعد** ما الهجوم والدفاع كلهم جاهزين.

---


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))
sys.path.insert(0, str(Path().resolve().parent.parent / 'attack' / 'src'))

import pandas as pd
from generator import PhishingGenerator
from mutator import AdversarialMutator
from adversarial.src.evaluator import AdversarialEvaluator


## 1. توليد رسائل تصيد جديدة


In [7]:
df = pd.read_csv('../../data/processed/final_dataset.csv').drop_duplicates(subset='text')
gen = PhishingGenerator(seed=99)  # different seed for fresh messages
gen.fit(df[df['Label']==0]['text'].tolist())

phishing_msgs = [m['text'] for m in gen.generate(count=50)]
print(f'Generated {len(phishing_msgs)} fresh phishing messages')


Generated 50 fresh phishing messages


## 2. اختبار ضد موديل الدفاع


In [8]:
evl = AdversarialEvaluator()
report = evl.evaluate(phishing_msgs)

print(f'=== Defense Model Results ===')
print(f'  Tested:         {report["total"]}')
print(f'  Detected:       {report["detected"]}')
print(f'  Evaded:         {report["evaded"]}')
print(f'  Detection rate: {report["detection_rate"]*100:.1f}%')
print(f'  Evasion rate:   {report["evasion_rate"]*100:.1f}%')

if report['evaded_messages']:
    print(f'\n🚨 Evaded ({len(report["evaded_messages"])}):')
    for e in report['evaded_messages'][:5]:
        print(f'  → {e["text"][:80]}...')
else:
    print('\n✅ No evasions!')


=== Defense Model Results ===
  Tested:         50
  Detected:       50
  Evaded:         0
  Detection rate: 100.0%
  Evasion rate:   0.0%

✅ No evasions!


## 3. Adversarial Mutations


In [9]:
mut = AdversarialMutator(seed=99)
mut_results = mut.mutate_batch(phishing_msgs[:20], mutations_per_text=2)

print(f'Mutated {len(mut_results)} messages')
for r in mut_results[:3]:
    if r['changed']:
        print(f'\n  Mutations: {r["mutations_applied"]}')
        print(f'  Before: {r["original"][:70]}...')
        print(f'  After:  {r["mutated"][:70]}...')


Mutated 20 messages

  Mutations: ['zwsp']
  Before: FedEx: شحنتك رقم SA66591467 تأكيد هوية المستلم. أدخل بياناتك على fedex...
  After:  FedEx: شحنتك رقم SA66591467 تأكيد هوية المستلم. أدخل بياناتك على fede​...

  Mutations: ['shortener']
  Before: إشعار أمني: أكد هويتك عبر https://www.le6ara.com.sa/confirm شكراً لتعا...
  After:  إشعار أمني: أكد هويتك عبر http://goo.gl/1jnyt7 شكراً لتعاونكم....

  Mutations: ['noise']
  Before: رسالة من خدمة العملاء: عزيزي العميل، بطاقتك البنكية معلقة. اتصل فوراً ...
  After:  رسالة من خدمة العملاء: عزيزي العميل، بطاقتك البنكية‌ معلقة. اتصل فوراً...


In [10]:
mut_report = evl.evaluate_mutations(mut_results)
print(f'=== Mutation Effectiveness ===')
print(f'  Total evasions: {mut_report["total_evasions"]}/{mut_report["total_tested"]}')
print(f'  Evasion rate:   {mut_report["evasion_rate"]*100:.1f}%')

for m, s in mut_report['per_mutation_stats'].items():
    print(f'    {m:20s}: {s["evasions"]}/{s["total"]} ({s["evasion_rate"]*100:.0f}%)')


=== Mutation Effectiveness ===
  Total evasions: 1/20
  Evasion rate:   5.0%
    zwsp                : 0/5 (0%)
    shortener           : 0/3 (0%)
    noise               : 1/11 (9%)
    char_dup            : 0/2 (0%)
    reorder             : 1/6 (17%)
    synonym             : 0/2 (0%)


## ✅ النتائج

هذه النتائج تُستخدم لتحسين كل من:
- **الهجوم:** تطوير تقنيات تهرّب أقوى
- **الدفاع:** سد الثغرات المكتشفة

```
هجوم ←→ دفاع = تحسين مستمر
```
